# Visualização dos Resultados de Ranking — LIMIT

Lê os arquivos da pasta `outputs/ranking_eval` e gera gráficos interativos comparando modelos bi-encoder e cross-encoder por métrica e tipo de query.

In [1]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

OUTPUTS_DIR = Path("outputs/ranking_eval")
CHECKPOINTS_DIR = OUTPUTS_DIR / "checkpoints"

METRICS = ["MRR", "MAP", "Recall@5", "NDCG@5", "Recall@10", "NDCG@10"]

with open(OUTPUTS_DIR / "run_metadata_latest.json") as f:
    metadata = json.load(f)

print("Run metadata:")
print(json.dumps(metadata, indent=2))

Run metadata:
{
  "timestamp": "20260512_140207",
  "cross_models": [
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "cross-encoder/ms-marco-MiniLM-L-12-v2",
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "BAAI/bge-reranker-v2-m3"
  ],
  "bi_models": [
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-base",
    "ibm-granite/granite-embedding-311m-multilingual-r2"
  ],
  "k_values": [
    5,
    10
  ],
  "n_easy_negatives": 100,
  "max_queries": null,
  "n_rows_cross": 24,
  "n_rows_bi": 18,
  "n_rows_total": 42
}


In [2]:
# Carrega os CSVs principais
df_all   = pd.read_csv(OUTPUTS_DIR / "results_latest.csv")
df_bi    = pd.read_csv(OUTPUTS_DIR / "bi_results_latest.csv")
df_cross = pd.read_csv(OUTPUTS_DIR / "cross_results_latest.csv")

for df in [df_all, df_bi, df_cross]:
    df["model_short"] = df["model"].str.split("::").str[-1].str.split("/").str[-1]
    df["encoder_type"] = df["model"].str.split("::").str[0]

print(f"Linhas totais: {len(df_all)}  |  bi: {len(df_bi)}  |  cross: {len(df_cross)}")
display(df_all.head(3))

Linhas totais: 42  |  bi: 18  |  cross: 24


,model,query_type,MRR,MAP,Recall@5,NDCG@5,Recall@10,NDCG@10,model_short,encoder_type
0,cross::cross-encoder/ms-marco-MiniLM-L-6-v2,overall,0.527724,0.431414,0.512638,0.449073,0.642581,0.504085,ms-marco-MiniLM-L-6-v2,cross
1,cross::cross-encoder/ms-marco-MiniLM-L-6-v2,abstracao,0.244976,0.211929,0.329143,0.226358,0.428190,0.270415,ms-marco-MiniLM-L-6-v2,cross
2,cross::cross-encoder/ms-marco-MiniLM-L-6-v2,completude,0.549525,0.418923,0.440048,0.429918,0.612714,0.489553,ms-marco-MiniLM-L-6-v2,cross


In [3]:
# Carrega per-query de todos os modelos
per_query_frames = []

for csv_path in CHECKPOINTS_DIR.glob("bi_per_query_*.csv"):
    model_slug = csv_path.stem.replace("bi_per_query_", "").replace("__", "/")
    df_tmp = pd.read_csv(csv_path)
    df_tmp["model"] = model_slug
    df_tmp["encoder_type"] = "bi"
    per_query_frames.append(df_tmp)

for csv_path in CHECKPOINTS_DIR.glob("cross_per_query_*.csv"):
    model_slug = csv_path.stem.replace("cross_per_query_", "").replace("__", "/")
    df_tmp = pd.read_csv(csv_path)
    df_tmp["model"] = model_slug
    df_tmp["encoder_type"] = "cross"
    per_query_frames.append(df_tmp)

df_pq = pd.concat(per_query_frames, ignore_index=True)
df_pq["model_short"] = df_pq["model"].str.split("/").str[-1]
print(f"Total per-query rows: {len(df_pq)}")
display(df_pq.head(3))

Total per-query rows: 875


,sample_id,query_type,num_relevant,MRR,MAP,Recall@5,NDCG@5,Recall@10,NDCG@10,model,encoder_type,model_short
0,0,completude,6,0.333333,0.489220,0.500000,0.446854,0.833333,0.590647,intfloat/multilingual-e5-base,bi,multilingual-e5-base
1,0,expressao_booleana,4,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,intfloat/multilingual-e5-base,bi,multilingual-e5-base
2,0,contagem,3,0.500000,0.340909,0.333333,0.296082,0.666667,0.444123,intfloat/multilingual-e5-base,bi,multilingual-e5-base


## 1. Visão Geral — Comparação de Todos os Modelos (overall)

In [4]:
df_overall = df_all[df_all["query_type"] == "overall"].sort_values("MRR", ascending=False).copy()
colors = {"bi": "#4C9BE8", "cross": "#E8744C"}

fig = go.Figure()
for metric in METRICS:
    fig.add_trace(go.Bar(
        name=metric,
        x=df_overall["model_short"],
        y=df_overall[metric],
        visible=(metric == "MRR"),
        marker_color=[colors[t] for t in df_overall["encoder_type"]],
        text=df_overall[metric].round(3),
        textposition="outside",
    ))

buttons = [
    dict(label=m, method="update",
         args=[{"visible": [j == i for j in range(len(METRICS))]},
               {"yaxis.title.text": m}])
    for i, m in enumerate(METRICS)
]

fig.update_layout(
    title="Desempenho Overall por Modelo",
    xaxis_title="Modelo",
    yaxis_title="MRR",
    yaxis_range=[0, 1.05],
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0, y=1.15, showactive=True)],
    height=500,
)
fig.add_annotation(x=1.02, y=0.65, xref="paper", yref="paper", text="■ bi-encoder",     font=dict(color=colors["bi"]),    showarrow=False)
fig.add_annotation(x=1.02, y=0.60, xref="paper", yref="paper", text="■ cross-encoder",  font=dict(color=colors["cross"]), showarrow=False)
fig.show()

## 2. Radar — Perfil de Métricas por Modelo

In [5]:
df_radar = df_all[df_all["query_type"] == "overall"].copy()
palette = px.colors.qualitative.Plotly

fig = go.Figure()
for idx, (_, row) in enumerate(df_radar.iterrows()):
    values = [row[m] for m in METRICS] + [row[METRICS[0]]]
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=METRICS + [METRICS[0]],
        fill="toself",
        name=row["model_short"],
        opacity=0.7,
        line_color=palette[idx % len(palette)],
    ))

fig.update_layout(
    title="Radar de Métricas — Overall",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=550,
)
fig.show()

## 3. Heatmap — Métricas × Tipos de Query por Modelo

In [6]:
metric_heatmap = "NDCG@10"

for enc_type, df_enc in [("bi-encoder", df_bi), ("cross-encoder", df_cross)]:
    pivot = df_enc.pivot_table(index="model_short", columns="query_type", values=metric_heatmap)
    cols = ["overall"] + [c for c in pivot.columns if c != "overall"]
    pivot = pivot[cols]

    fig = px.imshow(
        pivot,
        text_auto=".3f",
        color_continuous_scale="Blues",
        zmin=0, zmax=1,
        title=f"Heatmap {metric_heatmap} — {enc_type}",
        aspect="auto",
        height=300 + 60 * len(pivot),
    )
    fig.update_xaxes(tickangle=-30)
    fig.show()

## 4. Barras Agrupadas — Métricas por Tipo de Query

In [7]:
metric_bar = "NDCG@10"

df_qt = df_all[df_all["query_type"] != "overall"].copy()

fig = px.bar(
    df_qt.sort_values(["query_type", metric_bar], ascending=[True, False]),
    x="query_type",
    y=metric_bar,
    color="model_short",
    barmode="group",
    title=f"{metric_bar} por Tipo de Query",
    labels={"query_type": "Tipo de Query", metric_bar: metric_bar, "model_short": "Modelo"},
    height=500,
    color_discrete_sequence=px.colors.qualitative.Plotly,
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 5. Boxplot — Distribuição Per-Query por Modelo

In [8]:
metric_box = "NDCG@10"

fig = px.box(
    df_pq,
    x="model_short",
    y=metric_box,
    color="encoder_type",
    points="outliers",
    title=f"Distribuição Per-Query de {metric_box}",
    labels={"model_short": "Modelo", metric_box: metric_box, "encoder_type": "Tipo"},
    color_discrete_map={"bi": "#4C9BE8", "cross": "#E8744C"},
    height=500,
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 6. Violino — Distribuição Per-Query por Tipo de Query

In [9]:
metric_vio = "MRR"

fig = px.violin(
    df_pq[df_pq["query_type"] != "overall"],
    x="query_type",
    y=metric_vio,
    color="encoder_type",
    box=True,
    points=False,
    title=f"Distribuição {metric_vio} por Tipo de Query",
    labels={"query_type": "Tipo de Query", metric_vio: metric_vio, "encoder_type": "Tipo"},
    color_discrete_map={"bi": "#4C9BE8", "cross": "#E8744C"},
    height=500,
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 7. Evolução entre Runs

In [10]:
run_frames = []
for path in sorted(OUTPUTS_DIR.glob("results_*.csv")):
    if "latest" in path.name:
        continue
    ts = path.stem.replace("results_", "")
    df_tmp = pd.read_csv(path)
    df_tmp["timestamp"] = ts
    run_frames.append(df_tmp)

if run_frames:
    df_runs = pd.concat(run_frames, ignore_index=True)
    df_runs["model_short"] = df_runs["model"].str.split("::").str[-1].str.split("/").str[-1]
    df_runs_overall = df_runs[df_runs["query_type"] == "overall"]

    for metric_line in ["MRR", "NDCG@10"]:
        fig = px.line(
            df_runs_overall,
            x="timestamp",
            y=metric_line,
            color="model_short",
            markers=True,
            title=f"Evolução de {metric_line} por Run",
            labels={"timestamp": "Run", metric_line: metric_line, "model_short": "Modelo"},
            height=450,
        )
        fig.show()
else:
    print("Apenas uma run disponível — gráfico de evolução não aplicável.")

## 8. Tabela Resumo — Ranking Final dos Modelos

In [11]:
df_summary = (
    df_all[df_all["query_type"] == "overall"]
    .sort_values("NDCG@10", ascending=False)
    .reset_index(drop=True)
)
df_summary.index += 1
df_summary["Rank"] = df_summary.index

display(
    df_summary[["Rank", "model_short", "encoder_type"] + METRICS]
    .rename(columns={"model_short": "Modelo", "encoder_type": "Tipo"})
    .style
    .background_gradient(subset=METRICS, cmap="Blues")
    .format({m: "{:.4f}" for m in METRICS})
)

,Rank,Modelo,Tipo,MRR,MAP,Recall@5,NDCG@5,Recall@10,NDCG@10
1,1,bge-reranker-v2-m3,cross,0.7655,0.6863,0.7789,0.7082,0.9673,0.7866
2,2,granite-embedding-311m-multilingual-r2,bi,0.7406,0.6682,0.7528,0.6854,0.9524,0.7674
3,3,paraphrase-multilingual-MiniLM-L12-v2,bi,0.6899,0.5988,0.6448,0.5953,0.9242,0.7108
4,4,multilingual-e5-base,bi,0.6675,0.6031,0.6798,0.6137,0.8710,0.6930
5,5,ms-marco-MiniLM-L-6-v2,cross,0.5277,0.4314,0.5126,0.4491,0.6426,0.5041
6,6,ms-marco-MiniLM-L-12-v2,cross,0.5007,0.4107,0.4975,0.4278,0.6388,0.4891
7,7,mmarco-mMiniLMv2-L12-H384-v1,cross,0.5069,0.4082,0.4562,0.4129,0.5918,0.4693
